# Mobilité des candidats Data Scientists : classification explicable

**Objectif** : prédire si un candidat cherche un nouvel emploi (`target = 1`), comparer cinq modèles et expliquer les prédictions avec SHAP et LIME, globalement et pour un individu, puis examiner l'équité (genre, localisation).

Plan : 1. Données · 2. Préparation · 3. Modélisation · 4. Évaluation · 5. Explicabilité globale · 6. Explicabilité locale (SHAP vs LIME) · 7. Équité · 8. Analyse critique

In [ ]:
# !pip install -q kagglehub shap lime xgboost scikit-learn pandas matplotlib seaborn
import warnings, time, glob
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import kagglehub, shap
from lime.lime_tabular import LimeTabularExplainer

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_score)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, roc_curve, precision_recall_curve,
                             ConfusionMatrixDisplay)
from xgboost import XGBClassifier

SEED = 42
sns.set_theme(style="whitegrid")

## 1. Données
Téléchargement automatique via `kagglehub` (identifiants Kaggle nécessaires si le jeu n'est pas déjà en cache).

In [ ]:
path = kagglehub.dataset_download("arashnic/hr-analytics-job-change-of-data-scientists")
csv = glob.glob(f"{path}/**/aug_train.csv", recursive=True)[0]
df = pd.read_csv(csv)
print(df.shape)
df.head()

In [ ]:
print(df.dtypes, "\n")
print("Valeurs manquantes (%) :")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False), "\n")
print("Proportion des classes :")
print(df["target"].value_counts(normalize=True).round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(),
                   ["education_level", "relevent_experience", "company_size", "gender"]):
    df.fillna({col: "inconnu"}).groupby(col)["target"].mean().sort_values().plot.barh(ax=ax)
    ax.set_title(f"Taux de la classe positive par {col}")
    ax.set_xlabel("Taux de recherche d'emploi")
plt.tight_layout(); plt.show()

sns.histplot(data=df, x="city_development_index", hue="target", stat="density",
             common_norm=False, bins=30)
plt.title("Indice de développement de la ville selon la cible"); plt.show()

sns.heatmap(df[["city_development_index", "training_hours", "target"]].corr(),
            annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Corrélations (variables numériques)"); plt.show()

## 2. Préparation
- `enrollee_id` supprimé (identifiant) ; `city` conservée.
- Numériques (`city_development_index`, `training_hours`) : imputation par la médiane, standardisation.
- Catégorielles : valeurs manquantes remplacées par la modalité « inconnu », encodage one-hot, catégories rares (< 1 %) regroupées.
- Découpage 80 / 20 stratifié, graine 42.

In [ ]:
data = df.drop(columns=["enrollee_id"])
X = data.drop(columns="target")
y = data["target"].astype(int)

num_cols = ["city_development_index", "training_hours"]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(len(X_train), len(X_test))

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="inconnu")),
                      ("oh", OneHotEncoder(min_frequency=0.01,
                                           handle_unknown="infrequent_if_exist",
                                           sparse_output=False))]), cat_cols),
], verbose_feature_names_out=False)

n_feat = preprocess.fit(X_train).get_feature_names_out().shape[0]
print("Nombre de variables après encodage :", n_feat)

## 3. Modélisation et comparaison
Cinq modèles, hyperparamètres optimisés par recherche aléatoire (validation croisée stratifiée à 5 plis, score AUC-ROC). Le déséquilibre de classes est traité par pondération. Le temps d'entraînement (recherche incluse) est mesuré pour chaque modèle.

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

candidates = {
    "Régression logistique": (
        LogisticRegression(max_iter=3000, class_weight="balanced"),
        {"clf__C": np.logspace(-3, 2, 20)}),
    "Arbre de décision": (
        DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
        {"clf__max_depth": [3, 5, 7, 10], "clf__min_samples_leaf": [5, 20, 50]}),
    "k plus proches voisins": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [5, 11, 21, 41], "clf__weights": ["uniform", "distance"]}),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [6, 10, None],
         "clf__min_samples_leaf": [1, 5, 10]}),
    "XGBoost": (
        XGBClassifier(scale_pos_weight=pos_weight, eval_metric="logloss",
                      n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [3, 5, 7],
         "clf__learning_rate": [0.03, 0.1], "clf__subsample": [0.8, 1.0]}),
}

fitted, fit_time = {}, {}
for name, (est, grid) in candidates.items():
    pipe = Pipeline([("prep", clone(preprocess)), ("clf", est)])
    search = RandomizedSearchCV(pipe, grid, n_iter=8, scoring="roc_auc",
                                cv=cv, random_state=SEED, n_jobs=-1)
    t0 = time.perf_counter()
    search.fit(X_train, y_train)
    fit_time[name] = time.perf_counter() - t0
    fitted[name] = search.best_estimator_
    print(f"{name:24s} AUC CV = {search.best_score_:.3f} | {fit_time[name]:6.1f} s | {search.best_params_}")

## 4. Évaluation
Jeu de test, seuil de décision 0,5. Le jeu étant déséquilibré (≈ 25 % de classe positive), la PR-AUC complète l'AUC-ROC.

In [ ]:
rows, proba = [], {}
for name, model in fitted.items():
    p = model.predict_proba(X_test)[:, 1]
    proba[name] = p
    pred = (p >= 0.5).astype(int)
    rows.append({"Modèle": name,
                 "AUC-ROC": roc_auc_score(y_test, p),
                 "PR-AUC": average_precision_score(y_test, p),
                 "Précision": precision_score(y_test, pred),
                 "Rappel": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "Temps (s)": fit_time[name]})
results = pd.DataFrame(rows).set_index("Modèle").sort_values("AUC-ROC", ascending=False)
results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, p in proba.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(title="Courbes ROC", xlabel="FPR", ylabel="TPR")
ax[1].axhline(y_test.mean(), color="k", ls="--")
ax[1].set(title="Courbes précision-rappel", xlabel="Rappel", ylabel="Précision")
ax[0].legend(); ax[1].legend(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax_, name in zip(axes, ["XGBoost", "Régression logistique"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, (proba[name] >= 0.5).astype(int), ax=ax_, colorbar=False)
    ax_.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# Effet du seuil de décision (modèle recommandé : régression logistique)
p = proba["Régression logistique"]
ths = np.linspace(0.05, 0.95, 91)
plt.plot(ths, [precision_score(y_test, p >= t, zero_division=0) for t in ths], label="Précision")
plt.plot(ths, [recall_score(y_test, p >= t) for t in ths], label="Rappel")
plt.axvline(0.5, color="gray", ls=":")
plt.xlabel("Seuil de décision"); plt.legend()
plt.title("Effet du seuil sur la précision et le rappel"); plt.show()

In [ ]:
# L'écart XGBoost / régression logistique est-il stable ? Validation croisée sur l'ensemble des données
cv_scores = {}
for name in ["XGBoost", "Régression logistique"]:
    cv_scores[name] = cross_val_score(clone(fitted[name]), X, y, cv=cv,
                                      scoring="roc_auc", n_jobs=1)
    print(f"{name:24s} AUC = {cv_scores[name].mean():.3f} ± {cv_scores[name].std():.3f}")
diff = cv_scores["XGBoost"] - cv_scores["Régression logistique"]
print(f"\nÉcart d'AUC par pli (XGBoost - LR) : {np.round(diff, 4)}")
print(f"Écart moyen = {diff.mean():.4f} ± {diff.std():.4f}")

## 5. Explicabilité globale
Importance propre au modèle, importance par permutation (comparable entre modèles) et SHAP.

In [ ]:
def parts(name):
    pipe = fitted[name]
    return pipe.named_steps["prep"], pipe.named_steps["clf"]

pre_x, xgb = parts("XGBoost")
pre_l, lr = parts("Régression logistique")
feat_names = pre_x.get_feature_names_out()

Xtr_t = pd.DataFrame(pre_x.transform(X_train), columns=feat_names)
Xte_t = pd.DataFrame(pre_x.transform(X_test), columns=feat_names)

# Importance propre au modèle
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
pd.Series(xgb.feature_importances_, index=feat_names).nlargest(12).sort_values().plot.barh(
    ax=ax[0], title="XGBoost : importance (gain)")
pd.Series(lr.coef_[0], index=feat_names).reindex(
    pd.Series(lr.coef_[0], index=feat_names).abs().nlargest(12).index).sort_values().plot.barh(
    ax=ax[1], title="Régression logistique : coefficients")
plt.tight_layout(); plt.show()

In [ ]:
# Importance par permutation (pipeline complet, baisse d'AUC sur le jeu de test)
perm = {}
for name in ["Régression logistique", "Random Forest", "XGBoost"]:
    r = permutation_importance(fitted[name], X_test, y_test, scoring="roc_auc",
                               n_repeats=10, random_state=SEED, n_jobs=-1)
    perm[name] = pd.Series(r.importances_mean, index=X_test.columns)
pd.DataFrame(perm).sort_values("XGBoost").plot.barh(
    figsize=(8, 6), title="Importance par permutation (baisse d'AUC)")
plt.tight_layout(); plt.show()

In [ ]:
def shap_matrix(name, n=1500):
    """Valeurs SHAP (classe positive) sur un échantillon du jeu de test."""
    _, model = parts(name)
    Xs = Xte_t.sample(min(n, len(Xte_t)), random_state=SEED)
    if name == "Régression logistique":
        v = shap.LinearExplainer(model, Xtr_t)(Xs).values
    else:
        v = shap.TreeExplainer(model)(Xs, check_additivity=False).values
    if v.ndim == 3:
        v = v[:, :, 1]
    return Xs, v

shap_res = {n: shap_matrix(n) for n in ["XGBoost", "Régression logistique", "Random Forest"]}

Xs, v = shap_res["XGBoost"]
shap.summary_plot(v, Xs, max_display=15)

In [ ]:
# Dépendance SHAP pour les variables principales (XGBoost)
top3 = pd.Series(np.abs(v).mean(0), index=feat_names).nlargest(3).index
for f in top3:
    shap.dependence_plot(f, v, Xs, interaction_index=None)

## 6. Explicabilité locale : SHAP et LIME
Un même candidat du jeu de test est expliqué par SHAP (waterfall) et par LIME, puis les deux méthodes sont comparées.

In [ ]:
i = 0   # position du candidat dans le jeu de test (à modifier)
x_i = Xte_t.iloc[[i]]
print(f"Candidat {i} | classe réelle : {int(y_test.iloc[i])}")
for name in ["XGBoost", "Régression logistique"]:
    print(f"  {name:24s} probabilité = {proba[name][i]:.3f}  -> décision : "
          f"{'cherche un emploi' if proba[name][i] >= 0.5 else 'reste'}")

# Waterfall SHAP (XGBoost)
shap.plots.waterfall(shap.TreeExplainer(xgb)(x_i)[0], max_display=12)

# Waterfall SHAP (régression logistique)
shap.plots.waterfall(shap.LinearExplainer(lr, Xtr_t)(x_i)[0], max_display=12)

In [ ]:
lime_exp = LimeTabularExplainer(
    Xtr_t.values, feature_names=list(feat_names),
    class_names=["Reste", "Cherche un emploi"], mode="classification",
    random_state=SEED)

def local_compare(name, i, k=5):
    _, model = parts(name)
    x = Xte_t.iloc[i]
    if name == "Régression logistique":
        s = pd.Series(shap.LinearExplainer(model, Xtr_t)(Xte_t.iloc[[i]]).values[0], index=feat_names)
    else:
        sv = shap.TreeExplainer(model)(Xte_t.iloc[[i]], check_additivity=False).values[0]
        s = pd.Series(sv if sv.ndim == 1 else sv[:, 1], index=feat_names)
    e = lime_exp.explain_instance(x.values, model.predict_proba, num_features=10)
    l = pd.Series({feat_names[j]: w for j, w in e.as_map()[1]})
    top_s = s.abs().nlargest(k).index
    top_l = l.abs().nlargest(k).index
    common = [f for f in top_s if f in top_l]
    same_sign = [f for f in common if np.sign(s[f]) == np.sign(l[f])]
    print(f"{name} | candidat {i}")
    print(f"  Top {k} SHAP : {list(top_s)}")
    print(f"  Top {k} LIME : {list(top_l)}")
    print(f"  Variables communes : {len(common)}/{k} | de même signe : {len(same_sign)}/{len(common)}\n")
    return s, l

s_lr, l_lr = local_compare("Régression logistique", i)
s_xg, l_xg = local_compare("XGBoost", i)

exp_plot = lime_exp.explain_instance(Xte_t.iloc[i].values, lr.predict_proba, num_features=10)
exp_plot.as_pyplot_figure(); plt.tight_layout(); plt.show()

In [ ]:
# Phrase de synthèse générée automatiquement (régression logistique)
p_i = proba["Régression logistique"][i]
niveau = "élevée" if p_i >= 0.5 else "faible"
up = s_lr.nlargest(2).index.tolist()
down = s_lr.nsmallest(2).index.tolist()
print(f"La probabilité de recherche d'emploi est {niveau} ({p_i:.0%}), "
      f"principalement à cause de « {up[0]} » et « {up[1]} » ; "
      f"à l'inverse, « {down[0]} » et « {down[1]} » la réduisent.")

## 7. Équité : localisation et genre
Part de l'importance SHAP (moyenne des |SHAP|) portée par les variables de localisation (`city_development_index` et `city`) et par le genre, pour chaque modèle.

In [ ]:
loc_feats = [f for f in feat_names if f.startswith("city_")]
gen_feats = [f for f in feat_names if f.startswith("gender_")]

rows = []
for name, (Xs_, v_) in shap_res.items():
    imp = pd.Series(np.abs(v_).mean(0), index=feat_names)
    rows.append({"Modèle": name,
                 "Localisation (%)": 100 * imp[loc_feats].sum() / imp.sum(),
                 "Genre (%)": 100 * imp[gen_feats].sum() / imp.sum()})
pd.DataFrame(rows).set_index("Modèle").round(1)

In [ ]:
# Comportement du modèle recommandé selon le genre (jeu de test, seuil 0,5)
g = X_test["gender"].fillna("inconnu").values
pred = (proba["Régression logistique"] >= 0.5).astype(int)
fair = pd.DataFrame({"genre": g, "y": y_test.values, "pred": pred})
tab = fair.groupby("genre").agg(effectif=("y", "size"),
                                taux_reel=("y", "mean"),
                                taux_predit_positif=("pred", "mean"))
tab["rappel"] = fair.groupby("genre").apply(
    lambda d: recall_score(d["y"], d["pred"]) if d["y"].sum() else np.nan)
tab.round(3)

## 8. Analyse critique

- **Cohérence SHAP / LIME** : comparer le nombre de variables communes dans le top 5 et la concordance des signes (section 6). Des signes concordants avec des amplitudes différentes indiquent des explications qualitativement cohérentes mais non interchangeables.
- **Valeurs manquantes comme signal** : les modalités « inconnu » (taille d'entreprise, discipline) figurent parmi les variables influentes ; le modèle exploite en partie l'absence d'information, qui reflète la collecte des données plus qu'un comportement réel.
- **Variables sensibles et équité** : la localisation porte une part importante de l'importance SHAP, le genre une part faible (section 7). Le modèle ne s'appuie pas directement sur le genre mais dépend fortement de l'origine géographique, d'où un risque de discrimination indirecte.
- **Performance contre interprétabilité** : l'écart d'AUC entre XGBoost et la régression logistique est faible (sections 4 et validation croisée) pour un coût d'entraînement bien supérieur ; la régression logistique est recommandée (coefficients lisibles, dépendance à la localisation auditable).
- **Actions pour le décideur** : utiliser le score pour prioriser des échanges avec les candidats, jamais pour les exclure ; auditer ou retirer les variables géographiques avant mise en production ; améliorer la collecte pour réduire les valeurs « inconnu ».
- **Limites** : données déclaratives et déséquilibrées, imputation par médiane et « inconnu », pas de validation temporelle, explications SHAP/LIME descriptives et non causales.